# WordPiece
BERT(Bidirectional Encoder Representations from Transformers):

保留常见的基础词汇（如 "play"），而将复杂的生僻词或派生词拆分成有意义的子词（如 "unbelievable" 拆分成 "un", "##believ", "##able"）。这样既控制了词表大小，又解决了 OOV 问题，同时保留了充足的语义信息。

在查看 BERT 的分词结果时，你经常会看到带有 ## 前缀的词（例如 ##ing）。

* 没有 ## 的词：表示这是一个完整词汇的开头。
* 带有 ## 的词：表示这个子词依附于它前面的词，是前面词汇的延续。

## 训练

WordPiece 和另一种子词算法 BPE（Byte-Pair Encoding）非常相似，都是自底向上合并的贪心算法，但它们的合并标准不同。BPE 是单纯合并出现频率最高的相邻子词对；而 WordPiece 是合并能让训练数据似然度提升最大的子词对。具体步骤如下：

1. 初始化： 将训练语料库中的所有词汇拆分成单个字符，并加入特殊的控制符（如 [CLS], [SEP], [UNK]），形成初始词表。此时除了单词首字母外，其余字母都加上 `##` 前缀。
2. 建立语言模型： 基于当前的词表对语料库建立一元语言模型（Unigram Language Model）。
3. 计算得分： 遍历语料库中所有相邻的子词对（例如子词 $A$ 和子词 $B$），计算合并它们带来的收益。WordPiece 的评分核心实际上是计算两个子词的互信息（PMI）：
   $$Score = \frac{P(AB)}{P(A) \cdot P(B)}$$
   其中 $P(AB)$ 是 $A$ 和 $B$ 组合在一起出现的概率，$P(A)$ 和 $P(B)$ 是它们单独出现的概率。这个公式的直观意义是：如果两个子词经常一起出现，且很少单独出现（即 $P(AB)$ 很大，而 $P(A)$ 和 $P(B)$ 很小），那么合并它们的得分就极高。
4. 合并： 挑选得分最高的子词对，将其合并成一个新的子词（如 $AB$），并加入词表中。
5. 循环： 重复步骤 2 到 4，直到词表达到预设的大小（例如 BERT 的词表大小为 30,522），或者合并操作带来的收益低于某个阈值。

## 推理

当模型在实际应用中遇到一段新文本时, WordPiece 使用的是最长匹配原则（Maximum Forward Matching）。假设我们遇到一个词：`unaffable`:

1. 算法会从左到右扫描，寻找词表中能匹配到的最长的前缀。假设它在词表中找到了 "un"。
2. 接着处理剩下的部分 "affable"。由于这不是词的开头，算法会寻找带有 ## 的最长匹配串。假设它找到了 "##aff"。
3. 继续处理剩下的 "able"。算法在词表中找到了 "##able"。
4. 最终，"unaffable" 被切分为 ["un", "##aff", "##able"]。

如果算法在任何一步找不到任何可以匹配的子词（即使退化到单个字符也找不到），整个原始单词就会被无情地标记为 [UNK]（在英文中极为罕见，但在处理多语言或特殊符号时可能发生）。

## 实现

### 第一步：准备数据与基础构建

In [21]:
import re
import torch
from collections import defaultdict
from datasets import load_dataset

# 1. 使用 Hugging Face 非流式数据集（一次性加载 split）
dataset = load_dataset(
    "wikitext",
    "wikitext-2-raw-v1",
    split="train",
)

# 2. 从完整数据集中抽样，构造小语料
max_samples = 500
corpus = []

for text in dataset["text"]:
    text = (text or "").strip()
    if not text:
        continue
    text = re.sub(r"[^a-zA-Z\s]", " ", text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    if text:
        corpus.append(text)
    if len(corpus) >= max_samples:
        break

# 3. 预分词（Pre-tokenization）：按空格切分单词，并统计词频
word_freqs = defaultdict(int)
for text in corpus:
    words = text.split()
    for word in words:
        word_freqs[word] += 1

print(f"非流式采样完成，句子数: {len(corpus)}")
print(f"原始 train split 样本数: {len(dataset)}")
print("基础词频统计(前20项):", dict(list(word_freqs.items())[:20]))

非流式采样完成，句子数: 500
原始 train split 样本数: 36718
基础词频统计(前20项): {'valkyria': 54, 'chronicles': 39, 'iii': 17, 'senj': 5, 'no': 29, 'unrecorded': 1, 'japanese': 6, 'lit': 4, 'of': 1120, 'the': 2646, 'battlefield': 8, 'commonly': 2, 'referred': 4, 'to': 775, 'as': 283, 'outside': 13, 'japan': 4, 'is': 141, 'a': 680, 'tactical': 3}


### 第二步：初始化字符级词表与拆分

WordPiece 的起点是将所有单词打碎成单个字母。除了首字母，其他字母都要加上 ## 前缀。

In [22]:
# 初始化词表（包含基础字符）
vocab = set()
# 记录每个单词当前的拆分状态
splits = {}

for word in word_freqs.keys():
    # 首字母不加 ##，后续字母加 ##
    split = [word[0]] + ["##" + c for c in word[1:]]
    splits[word] = split
    vocab.update(split)

# 添加特殊 Token
special_tokens = ["<PAD>", "<UNK>", "<CLS>", "<SEP>"]
vocab.update(special_tokens)

print("初始词表大小:", len(vocab))
print("单词 'learning' 的初始拆分:", splits['learning'])

初始词表大小: 56
单词 'learning' 的初始拆分: ['l', '##e', '##a', '##r', '##n', '##i', '##n', '##g']


### 第三步：定义核心评分与合并逻辑
WordPiece 的核心是挑选得分最高的相邻子词对进行合并。我们使用的评分公式为：

$$Score = \frac{freq(A, B)}{freq(A) \cdot freq(B)}$$

注意：现实中为了防止分母过大导致长词无法合并，通常会结合频次阈值（在前面实现过），这里我们实现其最核心的互信息思想。

In [23]:
def compute_pair_scores(splits, word_freqs):
    """计算相邻子词对的得分"""
    pair_freqs = defaultdict(int)
    token_freqs = defaultdict(int)
    
    # 统计独立 token 频率和相邻 pair 频率
    for word, split in splits.items():
        freq = word_freqs[word]
        for i in range(len(split)):
            token_freqs[split[i]] += freq
            if i < len(split) - 1:
                pair_freqs[(split[i], split[i+1])] += freq
                
    # 计算 WordPiece 得分
    pair_scores = {}
    for pair, freq in pair_freqs.items():
        score = freq / (token_freqs[pair[0]] * token_freqs[pair[1]])
        pair_scores[pair] = score
        
    return pair_scores

def merge_pair(a, b, splits):
    """将拆分状态中的特定相邻对 (a, b) 合并为 ab"""
    for word, split in splits.items():
        if len(split) == 1:
            continue
        i = 0
        while i < len(split) - 1:
            if split[i] == a and split[i+1] == b:
                # 合并操作
                # 注意处理 ## 逻辑：如果 b 有 ##，合并后去掉 b 的 ##
                merged_token = a + b[2:] if b.startswith("##") else a + b
                split = split[:i] + [merged_token] + split[i+2:]
            else:
                i += 1
        splits[word] = split
    return splits

### 第四步：执行训练循环构建最终词表
我们设定一个目标词表大小，不断循环“打分 -> 寻找最高分 -> 合并”的过程。

In [ ]:
vocab_size = 1400  # 设定目标词表大小 

while len(vocab) < vocab_size:
    scores = compute_pair_scores(splits, word_freqs)
    if not scores:
        break
        
    # 找到得分最高的组合
    best_pair = max(scores, key=scores.get)
    
    # 执行合并
    splits = merge_pair(best_pair[0], best_pair[1], splits)
    
    # 将新生成的 token 加入词表
    new_token = best_pair[0] + best_pair[1][2:] if best_pair[1].startswith("##") else best_pair[0] + best_pair[1]
    vocab.add(new_token)

# 为了后续 PyTorch 使用，我们需要建立 Token 到 ID 的映射字典
token2id = {token: i for i, token in enumerate(sorted(vocab))}
print(f"训练完成！最终词表大小: {len(vocab)}")
print("\n部分词表展示:")
for token, tid in list(token2id.items())[:15]:
    print(f"{token}: {tid}")

训练完成！最终词表大小: 1400

部分词表展示:
##a: 0
##abcock: 1
##ack: 2
##acking: 3
##acqu: 4
##adnought: 5
##adow: 6
##aff: 7
##affi: 8
##affic: 9
##ainbows: 10
##ajor: 11
##aksmith: 12
##akthrough: 13
##amp: 14


### 第五步：实现推理阶段的分词器与 PyTorch 转换
模型训练好后，我们需要实现“最长前缀匹配”来对新句子进行分词，并将它们转化为 PyTorch 的 Tensor。

In [25]:
def encode_wordpiece(text, token2id):
    """最长前缀匹配分词，并转换为 PyTorch Tensor"""
    words = text.lower().split()
    encoded_tokens = []
    
    for word in words:
        start = 0
        while start < len(word):
            end = len(word)
            best_token = "<UNK>"
            
            # 从最长可能寻找匹配
            while start < end:
                sub_str = word[start:end]
                if start > 0:
                    sub_str = "##" + sub_str
                    
                if sub_str in token2id:
                    best_token = sub_str
                    break
                end -= 1
                
            if best_token == "<UNK>":
                # 如果找不到匹配，按照严格 WordPiece，整个词变为 UNK
                encoded_tokens = ["<UNK>"]
                break
            else:
                encoded_tokens.append(best_token)
                start = end
                
    # 转换为 ID
    input_ids = [token2id.get(token, token2id["<UNK>"]) for token in encoded_tokens]
    
    # === PyTorch 核心转化 ===
    # 将普通的 Python list 转换为 PyTorch 可以进行梯度计算的 Tensor
    tensor_ids = torch.tensor(input_ids, dtype=torch.long)
    
    return encoded_tokens, tensor_ids

# 测试一下我们自己写的模型！
test_sentence = "machine unlearning" # unlearning 是未见过的组合，但词根存在
tokens, tensor_output = encode_wordpiece(test_sentence, token2id)

print(f"测试句子: '{test_sentence}'")
print(f"分词结果: {tokens}")
print(f"PyTorch Tensor 输出: {tensor_output}")
print(f"Tensor 形状: {tensor_output.shape}")

测试句子: 'machine unlearning'
分词结果: ['m', '##a', '##ch', '##i', '##n', '##e', 'un', '##l', '##e', '##a', '##r', '##n', '##ing']
PyTorch Tensor 输出: tensor([1007,    0,   45,   81,  169,   66, 1286,  126,   66,    0,  235,  169,
          97])
Tensor 形状: torch.Size([13])
